In [1]:
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
from collections import defaultdict
import time
import numpy as np
import csv
import os

In [3]:
data = pd.read_csv('../data/scidcc/SciDCC.csv')
#data.head()

In [ ]:
from ollama import Client
from pydantic import BaseModel
import json


topics = ["Earthquakes", "Pollution", "Genetically Modified", "Hurricanes Cyclones", "Agriculture & Food", "Animals", 
          "Weather", "Endangered Animals", "Climate", "Ozone Holes", "Biology", "New Species", "Environment", 
          "Biotechnology", "Geography", "Microbes", "Extinction", "Zoology", "Geology", "Global Warming"]

client = Client(
    host='host',
)
model = 'llama3.3:70b'

output_file = "llama3.3:70b-temp0.1topp1seed31.jsonl"

MAX_RETRIES = 3
RETRY_DELAY = 5  # seconds

class StructuredOutput(BaseModel):
  body_label: list[str]

with open(output_file, "a", encoding="utf-8") as jsonfile:
    
    for row in data.itertuples(index=False):
        article_body = row[4]
        article_title = row[2]
        original_label = row[5]
        prompt = f"""
        You are a classifier. You are given an article (A) and a list of labels (L): 
        
        A: {article_body}
        
        L: {topics}
        
        Choose ONE label from the list of labels (L), which matches the article's topic best. 
        Answer ONLY with the chosen label in JSON format. Do not produce any other output."""
        
        messages = [{'role': 'user','content': prompt}]
        retries = 0
        while True:
            try:
                response = client.chat(model=model, 
                                       messages=messages, 
                                       options={"temperature":0.1,
                                                "seed":31,
                                                #"top_k":10,
                                                "top_p":1,
                                                },
                                        format=StructuredOutput.model_json_schema(),
                                        )
                predicted = StructuredOutput.model_validate_json(response.message.content)
                break
            
            except Exception as e:
                retries += 1
                print(f"Error: {e} | Retrying {retries}/{MAX_RETRIES}")

                if retries >= MAX_RETRIES:
                    raise

                time.sleep(RETRY_DELAY)

        entry = {
            "original_label":original_label,
            "prediction":predicted.model_dump(),
            "article_title:":article_title,
            "article_body":article_body
        }
        jsonfile.write(json.dumps(entry, ensure_ascii=False) +"\n")
        jsonfile.flush()

print("All complete :)")


All complete :)
